# 0203 Merged Prediction Grid And Training Covariate Extraction

This notebook keeps the completed prediction-grid workflow and adds a matching training-dataset extraction workflow. Both workflows use the same covariate groups, the same active buffer rings, the same EPSG:4326 approximate degree buffers, the same mainland/no-lakes raster masking, and the same on-the-fly Hansen calculations.

## Expected Inputs And Objects

Before running this notebook, confirm that `PROJECT_DIR` points to the `KSPH Code` repo root that contains the `data`, `config`, and `R_python_code` folders.

### Training Point Dataset

The training extraction section expects `data/dataset1.csv`. This file is created by the `01_make_dataset1.R` script and should contain one row per presence or pseudo-absence point-year. Required columns are:

- `id`: unique row identifier used to merge the separate covariate exports back together.
- `year`: integer year for the row. The active workflow expects annual values from 2001-2025.
- `latitude` and `longitude`: WGS84 decimal-degree coordinates, EPSG:4326.
- `outcome`: `1` for an event/presence row and `0` for a pseudo-absence/control row.
- `type`: event-source/type label. This is retained for downstream filtering, but it is not used during covariate extraction.
- `country`: country label if already available. The R modeling scripts can also add/fill country later for the prediction grid.

The notebook builds Earth Engine point features from this table, creates saved donut-buffer table assets for the active buffer rings, exports covariate tables to Google Drive, and then merges those tables into `data/dataset2.csv`.

### Prediction Grid Dataset

The prediction-grid section creates or reads `data/prediction_grid_10km.csv`, with one row per grid-cell center. Expected columns are `grid_id`, `grid_batch`, `x`, `y`, `longitude`, and `latitude`. The final prediction-grid covariate table is saved as `data/prediction_grid_covariates_2020_2025.csv`; it contains one row per grid cell per prediction year.

### Study Area And Masking

The active study area is configured with `STUDY_AREA_FILE` and optional `STUDY_AREA_BBOX`. The current default points to `config/africacountries_nolakes.shp` and clips it to mainland Africa within 10 degrees of the equator. To run a country-specific analysis, point `STUDY_AREA_FILE` to that shapefile and set `STUDY_AREA_BBOX = None`; the workflow will use the full provided polygon. Rasters are masked outside the active study-area polygon, and buffers stay in EPSG:4326 approximate degree units. Buffers are not clipped to the country/lake polygon before reduction; raster masking controls which pixels contribute values.

### Earth Engine And Google Drive

You need an authenticated Google Earth Engine account with access to the configured `EE_PROJECT`, permission to create assets under the configured asset roots, and access to the public raster collections used by the helper code. The notebook stores buffer geometries as Earth Engine table assets, but Hansen rasters are computed on the fly and are not saved as image assets.

Google Drive for Desktop must be installed and syncing the configured Drive folders. Earth Engine writes CSV exports to Drive because this is much faster and more reliable than pulling large tables directly through Python. After each Drive export task finishes, Drive for Desktop syncs the CSVs to the local `G:/My Drive/...` folders used by the merge cells.

### Optional External Rasters

Optional external covariates are configured in `config/external_rasters.csv`. Local rasters must first be uploaded to Earth Engine as image assets. Then add a row with `covariate_name`, `gee_asset_id`, `buffer_mode` (`scaled` or `nonscaled`), `reducer`, `scale_m`, and `enabled = true`. Disabled rows or rows without a `gee_asset_id` are ignored.

### Active Covariate Outputs

The active scaled rings are `0_10km`, `10_25km`, and `25_50km`. Scaled covariate families are forest cover, same-year forest loss, 1-year lagged forest loss, 2-year lagged forest loss, forest fragmentation, and population density. Nonscaled covariates are extracted from the `0_10km` buffer and include precipitation, precipitation anomaly/z-score, temperature, temperature anomaly/z-score, PET, NDVI, NDVI anomaly/z-score, and elevation.

The prediction-grid section writes `data/prediction_grid_covariates_2020_2025.csv` and `data/prediction_grid_10km.csv`. The training section reads `data/dataset1.csv`, submits matching Drive exports for years 2001-2025, and merges them into `data/dataset2.csv` after Google Drive for Desktop syncs the CSVs locally.

The prediction-grid Drive export switches are off by default in this combined notebook because those exports have already been completed. The training section has its own switches, asset root, Drive folder, and export prefix so the two workflows do not overwrite each other.


In [17]:
from pathlib import Path
import platform
import sys
import warnings


def find_project_dir():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path('/Users/boazbaliejukia/Documents/GitHub/ESP/AnnualEbolaPrediction_EID/KSPH_Code'),
        Path(r'C:/Users/carso/OneDrive - University of North Carolina at Chapel Hill/CDC Work Docs/4ES Contracting/KSPH Code'),
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'R_python_code').exists() and (candidate / 'config' / 'predictor_list.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the KSPH Code repo root. Open this notebook from the repo root or from R_python_code.')


def google_drive_folder_candidates(folder_name):
    """Candidate local Google Drive for Desktop paths for a My Drive folder."""
    candidates = []
    if platform.system() == 'Windows':
        candidates.append(Path(r'G:/My Drive') / folder_name)
    else:
        cloud = Path.home() / 'Library' / 'CloudStorage'
        if cloud.exists():
            for drive_root in sorted(cloud.glob('GoogleDrive-*')):
                candidates.append(drive_root / 'My Drive' / folder_name)
        candidates.append(Path.home() / 'Google Drive' / 'My Drive' / folder_name)
        candidates.append(Path.home() / 'Google Drive' / folder_name)
    return candidates


def resolve_google_drive_folder(folder_name, required=False):
    """Return the local Google Drive for Desktop path for a My Drive folder.

    Windows: G:/My Drive/<folder>
    macOS: ~/Library/CloudStorage/GoogleDrive-*/My Drive/<folder>
           or ~/Google Drive/My Drive/<folder>

    If the folder is missing and required=False, return the preferred expected path
    and warn. If required=True, raise FileNotFoundError.
    """
    candidates = google_drive_folder_candidates(folder_name)
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    preferred = candidates[0] if candidates else Path.home() / 'Google Drive' / 'My Drive' / folder_name
    searched = '\n  - '.join(str(path) for path in candidates) if candidates else '(none)'
    message = (
        f'Could not find Google Drive folder "{folder_name}".\n'
        'Install Google Drive for Desktop, sign in with the same Google account used for Earth Engine exports,\n'
        'create/sync the folder under My Drive, then re-run this cell. Searched:\n'
        f'  - {searched}'
    )
    if required:
        raise FileNotFoundError(message)
    warnings.warn(message + f'\nUsing expected path for later checks: {preferred}')
    return preferred


def require_google_drive_folder(path, folder_name):
    if path.exists():
        return path
    raise FileNotFoundError(
        f'Google Drive sync folder is missing: {path}\n'
        f'Install Google Drive for Desktop and sync My Drive/{folder_name} before merge/export.'
    )


PROJECT_DIR = find_project_dir()
PYTHON_DIR = PROJECT_DIR / 'R_python_code'
OUTPUT_CSV = PROJECT_DIR / 'data' / 'prediction_grid_covariates_2020_2025.csv'
OUTPUT_GRID_CSV = PROJECT_DIR / 'data' / 'prediction_grid_10km.csv'
(PROJECT_DIR / 'data').mkdir(exist_ok=True)
(PROJECT_DIR / 'outputs').mkdir(exist_ok=True)

# Active study area. Set STUDY_AREA_BBOX = None to use the full extent/polygon
# of STUDY_AREA_FILE, for example when using a country-specific shapefile.
STUDY_AREA_NAME = 'equatorial_africa'
STUDY_AREA_FILE = PROJECT_DIR / 'config' / 'africacountries_nolakes.shp'
STUDY_AREA_BBOX = (-15.5, -10.0, 51.0, 10.0)
STUDY_AREA_CONTEXT_BUFFER_DEGREES = 1.0

# Earth Engine compute/billing project. Assets may still live in a different project.
# EE_PROJECT = 'ksphcollab-506218' # ran out of compute resources, switched to EqAfricaClimate project
# EE_PROJECT_NUMBER = '850406846857'
# EE_PROJECT = 'eqafricaclimate' # team project with more compute (needs IAM access)
# EE_PROJECT_NUMBER = '260949660003'

# Personal GEE project (Community Tier). Use this until team project access is granted.
EE_PROJECT = 'oceanic-column-504522-h3'
EE_PROJECT_NUMBER = None  # optional; set from GCP if you want it printed accurately


FORCE_GEE_AUTH = False
GEE_AUTH_MODE = 'localhost'

# Google Drive for Desktop must sync this folder locally for the merge step.
DRIVE_EXPORT_FOLDER = 'Event_ENM_Prediction_Exports'
DOWNLOADED_EXPORT_DIR = resolve_google_drive_folder(DRIVE_EXPORT_FOLDER, required=False)
PREDGRID_EXPORT_PREFIX = 'predgrid_deg_drive'

# Prediction-grid assets under your personal GEE project (writable).
# Team originals lived under projects/flu-landscape/assets/FiloPrediction/...
SUPPORT_ASSET_ROOT = 'projects/oceanic-column-504522-h3/assets/FiloPrediction/Event_ENM_support_assets'
PREDICTION_BUFFER_ASSET_ROOT = 'projects/oceanic-column-504522-h3/assets/FiloPrediction/Event_ENM_prediction_grid_donut_buffers_simplified'
# In this workflow Hansen rasters are computed on the fly, not saved as image assets.
CREATE_HANSEN_IMAGE_ASSETS = False
STUDY_REGION_SIMPLIFY_TOLERANCE_M = 5000

# Approximate 10 km at the equator. The whole prediction workflow stays in EPSG:4326.
GRID_STEP_DEGREES = 0.09
PREDICTION_YEARS = list(range(2020, 2026)) # this means 2020-2025 inclusive

# Prediction-grid phase switches. Drive exports are off here because the prediction-grid extraction has already been completed.
RUN_GRID_EXPORT = True #False
CREATE_SIMPLIFIED_STUDY_REGION_ASSET = True #False
CREATE_BUFFER_ASSETS = True #False
RUN_GEE_SMOKE_TEST = True #False
RUN_DRIVE_EXPORTS = True #False
MERGE_DOWNLOADED_FILES = True

# Existing buffer assets should be overwritten after geometry-definition changes.
OVERWRITE_SIMPLIFIED_STUDY_REGION_ASSET = False
OVERWRITE_BUFFER_ASSETS = False
SKIP_EXISTING_DRIVE_EXPORTS = True
VALIDATE_DOWNLOADED_EXPORTS = True
FAIL_IF_SMOKE_TEST_BLANK = True
SMOKE_TEST_MAX_FEATURES = 2
SMOKE_TEST_GROUPS = ['forest_cover', 'nonscaled_precip']
SMOKE_TEST_RINGS = ['0_10km']

# One batch for this test. Increase later only if per-year exports still struggle.
PREDICTION_BATCH_COUNT = 1

# Per-year exports keep each Earth Engine table task lighter.
EXPORT_ALL_YEARS_TOGETHER = False

BUFFER_RINGS = ['0_10km', '10_25km', '25_50km']
GROUPS_TO_EXPORT = [
    'forest_cover',
    'forest_loss_same_year',
    'forest_loss_1yr_prior',
    'forest_loss_2yr_prior',
    'fragmentation',
    'population',
    'nonscaled_precip',
    'nonscaled_temp',
    'nonscaled_pet',
    'nonscaled_ndvi',
    'nonscaled_elevation',
]

PREDICTION_BASE_COLUMNS = ['grid_id', 'grid_batch', 'x', 'y', 'year', 'longitude', 'latitude']
PREDICTION_GRID_COLUMNS = ['grid_id', 'grid_batch', 'x', 'y', 'longitude', 'latitude']

# Training extraction settings. These reuse the prediction-grid covariate/ring definitions,
# but keep separate assets, Drive exports, and local outputs.
TRAINING_DATASET1_CSV = PROJECT_DIR / 'data' / 'dataset1.csv'
TRAINING_OUTPUT_CSV = PROJECT_DIR / 'data' / 'dataset2.csv'
TRAINING_BUFFER_ASSET_ROOT = 'projects/oceanic-column-504522-h3/assets/FiloPrediction/Event_ENM_training_degree_donut_buffers_simplified'
TRAINING_DRIVE_EXPORT_FOLDER = 'Event_ENM_Exports'
TRAINING_DOWNLOADED_EXPORT_DIR = resolve_google_drive_folder(TRAINING_DRIVE_EXPORT_FOLDER, required=False)
# Fresh prefix for this combined notebook so older standalone-02 exports are not merged by accident.
TRAINING_EXPORT_PREFIX = 'dataset2_0203_deg_drive'

# Short Mac end-to-end test year first; set to None for all years in dataset1 (2001-2025).
TRAINING_YEARS_TO_EXPORT = [2020]
TRAINING_BUFFER_RINGS = BUFFER_RINGS

# Training phase switches. You can leave these True and manually wait after asynchronous GEE cells.
# Do not use Run All from a fresh state because buffer assets must finish before smoke tests/exports,
# and Drive exports must finish and sync before the merge cell.
# First Mac run: create buffer assets once, then smoke-test before full Drive exports.
TRAINING_CREATE_BUFFER_ASSETS = True
TRAINING_RUN_GEE_SMOKE_TEST = True
TRAINING_RUN_DRIVE_EXPORTS = True #False
TRAINING_MERGE_DOWNLOADED_FILES = True

TRAINING_CREATE_ASSET_FOLDER = True
TRAINING_OVERWRITE_BUFFER_ASSETS = True #False
TRAINING_SKIP_EXISTING_DRIVE_EXPORTS = True
TRAINING_VALIDATE_DOWNLOADED_EXPORTS = True
TRAINING_CSV_READ_ATTEMPTS = 3
TRAINING_CSV_READ_WAIT_SECONDS = 10
TRAINING_FAIL_IF_SMOKE_TEST_BLANK = True
TRAINING_SMOKE_TEST_YEAR = 2020
TRAINING_SMOKE_TEST_MAX_FEATURES = 3
TRAINING_CLIP_BUFFERS_TO_STUDY_AREA = True #False
TRAINING_EXPORT_ALL_YEARS_TOGETHER = True
TRAINING_BASE_COLUMNS = ['id', 'year', 'latitude', 'longitude', 'outcome', 'type', 'country']
TRAINING_MERGE_KEY = 'id'

BUFFER_ASSET_MAX_VERTICES = 1_000_000
DRY_RUN = False

print(f'Python: {sys.executable}')
print(f'PROJECT_DIR: {PROJECT_DIR}')
print(f'Prediction Drive sync folder: {DOWNLOADED_EXPORT_DIR} (exists={DOWNLOADED_EXPORT_DIR.exists()})')
print(f'Training Drive sync folder: {TRAINING_DOWNLOADED_EXPORT_DIR} (exists={TRAINING_DOWNLOADED_EXPORT_DIR.exists()})')
print(f'TRAINING_CREATE_BUFFER_ASSETS={TRAINING_CREATE_BUFFER_ASSETS}')
print(f'TRAINING_RUN_GEE_SMOKE_TEST={TRAINING_RUN_GEE_SMOKE_TEST}')
print(f'TRAINING_RUN_DRIVE_EXPORTS={TRAINING_RUN_DRIVE_EXPORTS}')
print(f'TRAINING_YEARS_TO_EXPORT={TRAINING_YEARS_TO_EXPORT}')


Python: /Users/boazbaliejukia/Documents/GitHub/ESP/AnnualEbolaPrediction_EID/KSPH_Code/.venv/bin/python
PROJECT_DIR: /Users/boazbaliejukia/Documents/GitHub/ESP/AnnualEbolaPrediction_EID/KSPH_Code
Prediction Drive sync folder: /Users/boazbaliejukia/Library/CloudStorage/GoogleDrive-baliejukiabaliesima@gmail.com/My Drive/Event_ENM_Prediction_Exports (exists=False)
Training Drive sync folder: /Users/boazbaliejukia/Library/CloudStorage/GoogleDrive-baliejukiabaliesima@gmail.com/My Drive/Event_ENM_Exports (exists=False)
TRAINING_CREATE_BUFFER_ASSETS=True
TRAINING_RUN_GEE_SMOKE_TEST=True
TRAINING_RUN_DRIVE_EXPORTS=True
TRAINING_YEARS_TO_EXPORT=[2020]


/var/folders/yf/2d4jrrh95_dcgd8ps889nvy40000gn/T/ipykernel_7175/1891760801.py:62: UserWarning: Could not find Google Drive folder "Event_ENM_Prediction_Exports".
Install Google Drive for Desktop, sign in with the same Google account used for Earth Engine exports,
create/sync the folder under My Drive, then re-run this cell. Searched:
  - /Users/boazbaliejukia/Library/CloudStorage/GoogleDrive-baliejukiabaliesima@gmail.com/My Drive/Event_ENM_Prediction_Exports
  - /Users/boazbaliejukia/Google Drive/My Drive/Event_ENM_Prediction_Exports
  - /Users/boazbaliejukia/Google Drive/Event_ENM_Prediction_Exports
Using expected path for later checks: /Users/boazbaliejukia/Library/CloudStorage/GoogleDrive-baliejukiabaliesima@gmail.com/My Drive/Event_ENM_Prediction_Exports
  warnings.warn(message + f'\nUsing expected path for later checks: {preferred}')
/var/folders/yf/2d4jrrh95_dcgd8ps889nvy40000gn/T/ipykernel_7175/1891760801.py:62: UserWarning: Could not find Google Drive folder "Event_ENM_Exports"

In [18]:
import importlib.util
import re
import sys
from collections import defaultdict

import ee
import pandas as pd


def load_module_from_path(module_name, path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Could not load {module_name} from {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


covariates = load_module_from_path('covariates', PYTHON_DIR / '02_prediction_grid_helper.py')
if FORCE_GEE_AUTH:
    ee.Authenticate(force=True, auth_mode=GEE_AUTH_MODE)
covariates.initialize_earth_engine(EE_PROJECT)

selected_rings = [suffix for _, _, suffix in covariates.selected_scaled_rings(BUFFER_RINGS)]
external_specs = covariates.read_external_raster_specs()

print(f'Prediction years: {PREDICTION_YEARS}')
print(f'Grid step: {GRID_STEP_DEGREES} degrees (~{covariates.approx_degrees_to_m(GRID_STEP_DEGREES):,} m at the equator)')
print(f'Grid batches: {PREDICTION_BATCH_COUNT}')
print(f'Export all years together: {EXPORT_ALL_YEARS_TOGETHER}')
print('Hansen raster assets: disabled; Hansen covariates compute on the fly during table exports')
print(f'Rings: {selected_rings}')
for inner_deg, outer_deg, suffix in covariates.selected_scaled_rings(BUFFER_RINGS):
    tolerance_deg = covariates.buffer_geometry_error_degrees(outer_deg)
    scale_deg = covariates.ring_scale_degrees(suffix)
    print(
        f'  {suffix}: {inner_deg:.3f}-{outer_deg:.3f} degrees '
        f'(~{covariates.approx_degrees_to_km(inner_deg):.1f}-{covariates.approx_degrees_to_km(outer_deg):.1f} km) | '
        f'simplify tolerance {tolerance_deg:.4f} degrees (~{covariates.approx_degrees_to_m(tolerance_deg):,} m) | '
        f'Hansen degree scale {scale_deg:.4f} (~{covariates.approx_degrees_to_m(scale_deg):,} m)'
    )
print(f'Groups: {GROUPS_TO_EXPORT}')
print(f'GEE compute project: {EE_PROJECT}')
print(f'GEE project number: {EE_PROJECT_NUMBER}')
print(f'Study area name: {STUDY_AREA_NAME}')
print(f'Study area file: {STUDY_AREA_FILE}')
print(f'Study area bbox: {STUDY_AREA_BBOX if STUDY_AREA_BBOX is not None else "full provided polygon extent"}')
print(f'Study area context buffer: {STUDY_AREA_CONTEXT_BUFFER_DEGREES} degrees')
print(f'Support asset root: {SUPPORT_ASSET_ROOT}')
print(f'Study-region simplify tolerance: {STUDY_REGION_SIMPLIFY_TOLERANCE_M:,} m')
print(f'Prediction buffer asset root: {PREDICTION_BUFFER_ASSET_ROOT}')
print(f'Drive export folder: {DRIVE_EXPORT_FOLDER}')
print(f'Drive export prefix: {PREDGRID_EXPORT_PREFIX}')
print(f'Download/merge folder: {DOWNLOADED_EXPORT_DIR}')
print(f'Skip existing synced Drive CSVs: {SKIP_EXISTING_DRIVE_EXPORTS}')


Prediction years: [2020, 2021, 2022, 2023, 2024, 2025]
Grid step: 0.09 degrees (~10,019 m at the equator)
Grid batches: 1
Export all years together: False
Hansen raster assets: disabled; Hansen covariates compute on the fly during table exports
Rings: ['0_10km', '10_25km', '25_50km']
  0_10km: 0.000-0.090 degrees (~0.0-10.0 km) | simplify tolerance 0.0018 degrees (~200 m) | Hansen degree scale 0.0018 (~200 m)
  10_25km: 0.090-0.225 degrees (~10.0-25.0 km) | simplify tolerance 0.0045 degrees (~501 m) | Hansen degree scale 0.0045 (~501 m)
  25_50km: 0.225-0.449 degrees (~25.0-50.0 km) | simplify tolerance 0.0270 degrees (~3,006 m) | Hansen degree scale 0.0180 (~2,004 m)
Groups: ['forest_cover', 'forest_loss_same_year', 'forest_loss_1yr_prior', 'forest_loss_2yr_prior', 'fragmentation', 'population', 'nonscaled_precip', 'nonscaled_temp', 'nonscaled_pet', 'nonscaled_ndvi', 'nonscaled_elevation']
GEE compute project: oceanic-column-504522-h3
GEE project number: 260949660003
Study area name: 

In [19]:
SCALED_GROUP_BANDS = {
    'forest_cover': ['forest_cover_prop'],
    'forest_loss_same_year': ['flsy_prop'],
    'forest_loss_1yr_prior': ['fl1yp_prop'],
    'forest_loss_2yr_prior': ['fl2yp_prop'],
    'fragmentation': ['frag_edge_prop'],
    'population': ['pop_density'],
}

NONSCALED_EXPORT_GROUPS = {
    'nonscaled': None,
    'nonscaled_precip': {'era5_precip'},
    'nonscaled_temp': {'era5_temp'},
    'nonscaled_pet': {'era5_pet'},
    'nonscaled_ndvi': {'modis_ndvi'},
    'nonscaled_elevation': {'elevation'},
    'nonscaled_external': 'external',
}


def is_nonscaled_export_group(group_name):
    return group_name in NONSCALED_EXPORT_GROUPS


def mask_to_land(image):
    return image.updateMask(land_mask)


def nonscaled_image_groups_for_export(year, export_group):
    groups = covariates.nonscaled_covariate_image_groups(year, external_specs)
    selection = NONSCALED_EXPORT_GROUPS.get(export_group)
    if selection is None:
        selected = groups
    elif selection == 'external':
        selected = [group for group in groups if group.name.startswith('external_')]
    else:
        selected = [group for group in groups if group.name in selection]
    return [
        covariates.CovariateImageGroup(
            name=group.name,
            image=mask_to_land(group.image).toFloat(),
            band_names=group.band_names,
            scale_m=group.scale_m,
        )
        for group in selected
    ]


def selected_ring_tuple(ring_suffix):
    matches = [ring for ring in covariates.selected_scaled_rings([ring_suffix]) if ring[2] == ring_suffix]
    if not matches:
        raise ValueError(f'Unknown ring suffix: {ring_suffix}')
    return matches[0]


def make_prediction_grid(region, step_degrees, batch_count):
    step_degrees = float(step_degrees)
    batch_count = int(batch_count)
    projection = covariates.degree_projection(step_degrees)
    sampled = (
        ee.Image.pixelLonLat()
        .reproject(projection)
        .sample(
            region=region,
            projection=projection,
            geometries=True,
            tileScale=4,
        )
    )

    def add_grid_fields(feature):
        feature = ee.Feature(feature)
        lon = ee.Number(feature.get('longitude'))
        lat = ee.Number(feature.get('latitude'))
        geom = ee.Geometry.Point([lon, lat], covariates.LAT_LONG_CRS)
        grid_col = lon.subtract(-180).divide(step_degrees).floor()
        grid_row = ee.Number(90).subtract(lat).divide(step_degrees).floor()
        x = lon
        y = lat
        grid_batch = grid_col.abs().multiply(1000003).add(grid_row.abs()).mod(batch_count).toInt()
        grid_id = grid_col.format('%.0f').cat('_').cat(grid_row.format('%.0f'))
        return ee.Feature(geom, {
            'grid_id': grid_id,
            'grid_batch': grid_batch,
            'x': x,
            'y': y,
            'longitude': lon,
            'latitude': lat,
        })

    return sampled.map(add_grid_fields)


def grid_batch_ids():
    return list(range(int(PREDICTION_BATCH_COUNT)))


def grid_batch(grid, batch_id):
    return grid.filter(ee.Filter.eq('grid_batch', int(batch_id)))


def prediction_buffer_asset_id(ring_suffix, batch_id):
    return covariates.prediction_buffer_asset_id(
        PREDICTION_BUFFER_ASSET_ROOT,
        ring_suffix,
        batch_id=int(batch_id),
    )


def prediction_buffer_features_for_year(ring_suffix, batch_id, year, limit=None):
    features = ee.FeatureCollection(prediction_buffer_asset_id(ring_suffix, batch_id))
    if limit is not None:
        features = features.limit(int(limit))
    return features.map(lambda feature: ee.Feature(feature).set({'year': int(year)}))


def hansen_source_image(year, ring_suffix):
    return covariates.prediction_hansen_image(int(year), ring_suffix, study_region=land_mask_region)


def scaled_source_image(year, group_name, ring_suffix):
    year_number = ee.Number(int(year))
    if group_name in {'forest_cover', 'forest_loss_same_year', 'forest_loss_1yr_prior', 'forest_loss_2yr_prior', 'fragmentation'}:
        scale_m = covariates.scaled_family_scale_m('forest_loss' if group_name.startswith('forest_loss') else group_name, ring_suffix)
        hansen = hansen_source_image(year, ring_suffix)
    elif group_name == 'population':
        scale_m = covariates.scaled_family_scale_m('population', ring_suffix)
    else:
        raise ValueError(f'Unknown scaled group: {group_name}')

    if group_name == 'forest_cover':
        return hansen.select(['forest_cover_prop']), ['forest_cover_prop'], scale_m
    if group_name == 'forest_loss_same_year':
        return hansen.select(['flsy_prop']), ['flsy_prop'], scale_m
    if group_name == 'forest_loss_1yr_prior':
        return hansen.select(['fl1yp_prop']), ['fl1yp_prop'], scale_m
    if group_name == 'forest_loss_2yr_prior':
        return hansen.select(['fl2yp_prop']), ['fl2yp_prop'], scale_m
    if group_name == 'fragmentation':
        return hansen.select(['frag_edge_prop']), ['frag_edge_prop'], scale_m
    if group_name == 'population':
        return mask_to_land(covariates.landscan_population_density(year_number)), ['pop_density'], scale_m


def reduce_image_over_prediction_buffers(image, features, band_names, ring_suffix, scale_m):
    output_names = [f'{band}_{ring_suffix}' for band in band_names]
    collection = covariates.reduce_image_over_buffer_features(
        image=mask_to_land(image),
        features=features,
        band_names=band_names,
        ring_suffix=ring_suffix,
        scale_m=scale_m,
    )
    return collection, output_names


def extract_scaled_prediction_batch_year(year, group_name, ring_suffix, batch_id, limit=None):
    image, band_names, scale_m = scaled_source_image(year, group_name, ring_suffix)
    features = prediction_buffer_features_for_year(ring_suffix, batch_id, year, limit=limit)
    return reduce_image_over_prediction_buffers(image, features, band_names, ring_suffix, scale_m)


def extract_nonscaled_prediction_batch_year(year, group_name, batch_id, limit=None):
    groups = nonscaled_image_groups_for_export(ee.Number(int(year)), group_name)
    if not groups:
        raise ValueError(f'No nonscaled image groups matched {group_name}.')
    features = prediction_buffer_features_for_year('0_10km', batch_id, year, limit=limit)
    collections = []
    output_names = []
    for group in groups:
        collection, props = reduce_image_over_prediction_buffers(
            group.image,
            features,
            group.band_names,
            '0_10km',
            group.scale_m,
        )
        collections.append((collection, props))
        output_names.extend(props)
    if len(collections) == 1:
        return collections[0][0], output_names
    return covariates.merge_ring_feature_collections(collections, join_key='grid_id'), output_names


def merge_year_collections(collections):
    combined = ee.FeatureCollection([])
    for collection in collections:
        combined = combined.merge(collection)
    return combined


def normalized_export_prefix(path):
    stem = Path(path).stem
    return re.sub(r'-\d{10,}-\d{10,}$', '', stem)


def existing_drive_export_prefixes(download_dir):
    download_path = Path(download_dir)
    if not download_path.exists():
        print(f'Drive sync folder not found yet: {download_path}')
        return set()
    prefixes = set()
    for path in download_path.glob('*.csv'):
        prefix = normalized_export_prefix(path)
        if prefix.startswith(f'{PREDGRID_EXPORT_PREFIX}_') or prefix.startswith('prediction_grid_10km'):
            prefixes.add(prefix)
    return prefixes


def drive_description(group_name, ring_suffix=None, year=None, batch_id=None):
    year_part = 'all_years' if year is None else str(int(year))
    batch_part = '' if batch_id is None else f'_batch_{int(batch_id):03d}'
    if ring_suffix is None:
        return f'{PREDGRID_EXPORT_PREFIX}_{year_part}_{group_name}{batch_part}'
    return f'{PREDGRID_EXPORT_PREFIX}_{year_part}_{group_name}_{ring_suffix}{batch_part}'


def expected_drive_descriptions(groups, rings, years):
    names = []
    year_values = [None] if EXPORT_ALL_YEARS_TOGETHER else years
    for group in groups:
        if is_nonscaled_export_group(group):
            for batch_id in grid_batch_ids():
                for year in year_values:
                    names.append(drive_description(group, year=year, batch_id=batch_id))
        else:
            for ring in rings:
                for batch_id in grid_batch_ids():
                    for year in year_values:
                        names.append(drive_description(group, ring, year=year, batch_id=batch_id))
    return names


def summarize_nonnull_properties(label, collection, props):
    row_count = int(collection.size().getInfo())
    counts = {prop: int(collection.aggregate_count(prop).getInfo()) for prop in props}
    sample_values = {}
    for prop, count in counts.items():
        sample_values[prop] = None
        if count > 0:
            sample_values[prop] = collection.filter(ee.Filter.notNull([prop])).first().get(prop).getInfo()
    blank_props = [prop for prop, count in counts.items() if count == 0]
    print(f'{label}: rows={row_count}; non-null counts={counts}')
    print(f'  sample values={sample_values}')
    return {
        'export': label,
        'rows': row_count,
        'blank_columns': ','.join(blank_props),
        'sample_values': sample_values,
    }


## Step 1: Create Prediction Grid

This creates an approximate 10 km point grid over the active study area using a 0.09 degree EPSG:4326 grid. The active study area is the provided polygon, optionally clipped to `STUDY_AREA_BBOX`. The raster mask/support geometry uses the same polygon with `STUDY_AREA_CONTEXT_BUFFER_DEGREES` added around the bbox when a bbox is supplied, then it is simplified and optionally saved as a reusable GEE table asset.

The grid is generated inside Earth Engine in EPSG:4326. Coordinates are preserved as `x`/`y` lon/lat centers plus duplicate `longitude`/`latitude` columns for readability. `grid_batch` is used only to split larger Earth Engine exports into smaller tasks.


In [20]:
core_region = covariates.load_study_region(
    path=STUDY_AREA_FILE,
    bbox=STUDY_AREA_BBOX,
)
raw_land_mask_region = covariates.load_study_region(
    path=STUDY_AREA_FILE,
    bbox=STUDY_AREA_BBOX,
    lat_buffer_degrees=STUDY_AREA_CONTEXT_BUFFER_DEGREES,
    lon_buffer_degrees=STUDY_AREA_CONTEXT_BUFFER_DEGREES,
)
land_mask_region = covariates.simplify_study_region(raw_land_mask_region, STUDY_REGION_SIMPLIFY_TOLERANCE_M)
simplified_study_region_asset_id = covariates.simplified_study_region_asset_id(
    SUPPORT_ASSET_ROOT,
    STUDY_REGION_SIMPLIFY_TOLERANCE_M,
    study_area_name=STUDY_AREA_NAME,
)
land_mask = covariates.land_mask_image(land_mask_region)
prediction_grid = make_prediction_grid(core_region, GRID_STEP_DEGREES, PREDICTION_BATCH_COUNT)

if CREATE_SIMPLIFIED_STUDY_REGION_ASSET:
    study_region_rows = []
    study_region_rows.extend(covariates.ensure_ee_folder(SUPPORT_ASSET_ROOT, dry_run=DRY_RUN))
    study_region_rows.append(
        covariates.export_study_region_to_asset(
            study_region=land_mask_region,
            description=f'{STUDY_AREA_NAME}_simplified_{STUDY_REGION_SIMPLIFY_TOLERANCE_M}m',
            asset_id=simplified_study_region_asset_id,
            tolerance_m=STUDY_REGION_SIMPLIFY_TOLERANCE_M,
            study_area_name=STUDY_AREA_NAME,
            dry_run=DRY_RUN,
            overwrite=OVERWRITE_SIMPLIFIED_STUDY_REGION_ASSET,
        )
    )
    manifest_path = PROJECT_DIR / 'outputs' / 'gee_prediction_simplified_study_region_manifest.csv'
    covariates.write_manifest(study_region_rows, manifest_path)
    print(f'Wrote simplified study-region asset manifest: {manifest_path}')
    print(f'Simplified study-region asset: {simplified_study_region_asset_id}')
else:
    print('CREATE_SIMPLIFIED_STUDY_REGION_ASSET is False; using in-session simplified study region only.')

grid_count = int(prediction_grid.size().getInfo())
distinct_grid_count = int(prediction_grid.aggregate_count_distinct('grid_id').getInfo())
if distinct_grid_count != grid_count:
    raise ValueError(f'grid_id is not unique: {distinct_grid_count:,} distinct IDs for {grid_count:,} grid points.')

batch_counts = prediction_grid.aggregate_histogram('grid_batch').getInfo()
print(f'Prediction grid points: {grid_count:,}')
print(f'Distinct grid IDs: {distinct_grid_count:,}')
print('Batch counts:')
for batch_id in sorted(batch_counts, key=lambda value: int(value)):
    print(f'  batch {int(batch_id):03d}: {int(batch_counts[batch_id]):,} points')

sample_rows = prediction_grid.limit(5).getInfo()['features']
print('First grid rows:')
for feature in sample_rows:
    print(feature['properties'])


EEException: Permission 'earthengine.assets.create' denied on resource 'projects/flu-landscape' (or it may not exist).

## Step 2: Optional Grid Export

This standalone grid CSV is useful for checking grid coordinates and reconstructing rasters later. It is exported to Google Drive and will sync locally through Google Drive for Desktop.


In [8]:
if RUN_GRID_EXPORT:
    existing_prefixes = existing_drive_export_prefixes(DOWNLOADED_EXPORT_DIR) if SKIP_EXISTING_DRIVE_EXPORTS else set()
    description = 'prediction_grid_10km_polygon_buffers'
    if description in existing_prefixes:
        print(f'{description} | synced CSV exists; skipping grid export')
    else:
        manifest_row = covariates.export_table_to_drive(
            collection=prediction_grid,
            description=description,
            folder=DRIVE_EXPORT_FOLDER,
            selectors=PREDICTION_GRID_COLUMNS,
            dry_run=DRY_RUN,
        )
        manifest_path = PROJECT_DIR / 'outputs' / 'gee_prediction_grid_export_manifest.csv'
        covariates.write_manifest([manifest_row], manifest_path)
        print(f'Wrote grid export manifest: {manifest_path}')
else:
    print('RUN_GRID_EXPORT is False; skipping standalone grid export.')


RUN_GRID_EXPORT is False; skipping standalone grid export.


## Step 3: Create Prediction Buffer Assets

Run this if the prediction-grid buffer assets do not exist, or if you changed ring distances, simplification tolerances, the grid, or batch count.

The buffers are no longer intersected with the complex mainland/no-lakes polygon. They are simple circular/donut polygons, like the old QGIS workflow. Ocean and lake pixels are handled by masking all source rasters outside the mainland/no-lakes study region before reduction.

With the current one-batch test settings, this creates one table asset per ring: 3 buffer assets total. Existing assets are overwritten by default because the buffer geometry definition changed.

After submitting these tasks, pause until all asset exports finish in Earth Engine before running the smoke test or covariate exports.


In [9]:
if CREATE_BUFFER_ASSETS:
    buffer_rows = []
    buffer_rows.extend(covariates.ensure_ee_folder(PREDICTION_BUFFER_ASSET_ROOT, dry_run=DRY_RUN))

    for _, outer_deg, ring_suffix in covariates.selected_scaled_rings(BUFFER_RINGS):
        tolerance_deg = covariates.buffer_geometry_error_degrees(outer_deg)
        print(
            f'Buffer {ring_suffix}: outer radius {outer_deg:.3f} degrees '
            f'(~{covariates.approx_degrees_to_km(outer_deg):.1f} km); '
            f'simplify tolerance {tolerance_deg:.4f} degrees '
            f'(~{covariates.approx_degrees_to_m(tolerance_deg):,} m)'
        )
        for batch_id in grid_batch_ids():
            batch_points = grid_batch(prediction_grid, batch_id)
            buffers = covariates.make_prediction_buffer_collection(
                batch_points,
                ring_suffix,
                study_region=None,
            )
            buffer_rows.append(
                covariates.export_table_to_asset(
                    collection=buffers,
                    description=f'prediction_buffers_{ring_suffix}_batch_{batch_id:03d}',
                    asset_id=prediction_buffer_asset_id(ring_suffix, batch_id),
                    dry_run=DRY_RUN,
                    overwrite=OVERWRITE_BUFFER_ASSETS,
                    max_vertices=BUFFER_ASSET_MAX_VERTICES,
                )
            )

    manifest_path = PROJECT_DIR / 'outputs' / 'gee_prediction_buffer_asset_manifest.csv'
    covariates.write_manifest(buffer_rows, manifest_path)
    print(f'Wrote buffer asset manifest: {manifest_path}')
    print('Pause now: wait for all prediction buffer asset tasks to finish before continuing.')
else:
    print('CREATE_BUFFER_ASSETS is False; using existing prediction buffer assets.')


CREATE_BUFFER_ASSETS is False; using existing prediction buffer assets.


## Step 4: Hansen Raster Assets Disabled

This workflow does not save annual Hansen raster assets, because those assets take up Drive/asset storage. Hansen forest cover, forest loss, lagged forest loss, and fragmentation are computed on the fly during the table exports.

`CREATE_HANSEN_IMAGE_ASSETS = False` is kept near the top of the notebook as a visible reminder that this path is disabled.

With the intended settings, run this cell and it will simply confirm that raster-asset creation is skipped.


In [ ]:
if CREATE_HANSEN_IMAGE_ASSETS:
    raise ValueError('Hansen raster asset exports are disabled for this workflow. Keep CREATE_HANSEN_IMAGE_ASSETS = False.')

print('Skipping Hansen raster asset creation.')
print('Hansen variables will be computed on the fly during the table export tasks.')


## Step 5: Smoke Test Extraction

Run this after the prediction buffer asset tasks have finished. It reads the saved polygon buffer assets and checks a small sample before launching the full export set.


In [10]:
if RUN_GEE_SMOKE_TEST:
    smoke_year = int(PREDICTION_YEARS[0])
    smoke_batch_id = 0
    smoke_groups = list(SMOKE_TEST_GROUPS)
    smoke_rings = [ring for ring in SMOKE_TEST_RINGS if ring in selected_rings]
    if not smoke_rings:
        raise ValueError(f'No requested smoke-test rings are active: {SMOKE_TEST_RINGS}')
    print(f'Smoke test groups: {smoke_groups}')
    print(f'Smoke test rings: {smoke_rings}')
    print(f'Smoke test features per extract: {SMOKE_TEST_MAX_FEATURES}')
    smoke_rows = []
    for group_name in smoke_groups:
        if is_nonscaled_export_group(group_name):
            collection, props = extract_nonscaled_prediction_batch_year(
                smoke_year,
                group_name,
                smoke_batch_id,
                limit=SMOKE_TEST_MAX_FEATURES,
            )
            smoke_rows.append(
                summarize_nonnull_properties(
                    drive_description(group_name, year=smoke_year, batch_id=smoke_batch_id),
                    collection,
                    props,
                )
            )
        else:
            if group_name not in SCALED_GROUP_BANDS:
                raise ValueError(f'Unknown scaled group: {group_name}')
            for ring_suffix in smoke_rings:
                collection, props = extract_scaled_prediction_batch_year(
                    smoke_year,
                    group_name,
                    ring_suffix,
                    smoke_batch_id,
                    limit=SMOKE_TEST_MAX_FEATURES,
                )
                smoke_rows.append(
                    summarize_nonnull_properties(
                        drive_description(group_name, ring_suffix, smoke_year, batch_id=smoke_batch_id),
                        collection,
                        props,
                    )
                )

    smoke_summary = pd.DataFrame(smoke_rows)
    display(smoke_summary)
    failed = smoke_summary.loc[smoke_summary['blank_columns'].astype(str).ne('')]
    if FAIL_IF_SMOKE_TEST_BLANK and not failed.empty:
        raise ValueError('Smoke test found all-blank covariate columns. Review the table above before exporting.')
else:
    print('RUN_GEE_SMOKE_TEST is False; skipping pre-export smoke test.')


RUN_GEE_SMOKE_TEST is False; skipping pre-export smoke test.


## Step 6: Submit Prediction Covariate Drive Exports

This submits Earth Engine batch tasks. Exports are split by grid batch. After this cell finishes submitting tasks, pause and monitor task completion in Earth Engine. Google Drive for Desktop should sync completed CSVs into `DOWNLOADED_EXPORT_DIR`.


In [11]:
if RUN_DRIVE_EXPORTS:
    existing_prefixes = existing_drive_export_prefixes(DOWNLOADED_EXPORT_DIR) if SKIP_EXISTING_DRIVE_EXPORTS else set()
    if SKIP_EXISTING_DRIVE_EXPORTS:
        print(f'Found {len(existing_prefixes)} existing synced prediction export(s) to skip.')

    def should_submit_export(description):
        if description in existing_prefixes:
            print(f'{description} | synced CSV exists; skipping export')
            return False
        return True

    manifest_rows = []
    for group_name in GROUPS_TO_EXPORT:
        if is_nonscaled_export_group(group_name):
            for batch_id in grid_batch_ids():
                if EXPORT_ALL_YEARS_TOGETHER:
                    description = drive_description(group_name, batch_id=batch_id)
                    if not should_submit_export(description):
                        continue
                    year_collections = []
                    props = None
                    for year in PREDICTION_YEARS:
                        collection, props = extract_nonscaled_prediction_batch_year(year, group_name, batch_id)
                        year_collections.append(collection)
                    manifest_rows.append(
                        covariates.export_table_to_drive(
                            collection=merge_year_collections(year_collections),
                            description=description,
                            folder=DRIVE_EXPORT_FOLDER,
                            selectors=PREDICTION_BASE_COLUMNS + list(props or []),
                            dry_run=DRY_RUN,
                        )
                    )
                else:
                    for year in PREDICTION_YEARS:
                        description = drive_description(group_name, year=year, batch_id=batch_id)
                        if not should_submit_export(description):
                            continue
                        collection, props = extract_nonscaled_prediction_batch_year(year, group_name, batch_id)
                        manifest_rows.append(
                            covariates.export_table_to_drive(
                                collection=collection,
                                description=description,
                                folder=DRIVE_EXPORT_FOLDER,
                                selectors=PREDICTION_BASE_COLUMNS + list(props),
                                dry_run=DRY_RUN,
                            )
                        )
        else:
            if group_name not in SCALED_GROUP_BANDS:
                raise ValueError(f'Unknown scaled group: {group_name}')
            for ring_suffix in selected_rings:
                for batch_id in grid_batch_ids():
                    if EXPORT_ALL_YEARS_TOGETHER:
                        description = drive_description(group_name, ring_suffix, batch_id=batch_id)
                        if not should_submit_export(description):
                            continue
                        year_collections = []
                        props = None
                        for year in PREDICTION_YEARS:
                            collection, props = extract_scaled_prediction_batch_year(year, group_name, ring_suffix, batch_id)
                            year_collections.append(collection)
                        manifest_rows.append(
                            covariates.export_table_to_drive(
                                collection=merge_year_collections(year_collections),
                                description=description,
                                folder=DRIVE_EXPORT_FOLDER,
                                selectors=PREDICTION_BASE_COLUMNS + list(props or []),
                                dry_run=DRY_RUN,
                            )
                        )
                    else:
                        for year in PREDICTION_YEARS:
                            description = drive_description(group_name, ring_suffix, year=year, batch_id=batch_id)
                            if not should_submit_export(description):
                                continue
                            collection, props = extract_scaled_prediction_batch_year(year, group_name, ring_suffix, batch_id)
                            manifest_rows.append(
                                covariates.export_table_to_drive(
                                    collection=collection,
                                    description=description,
                                    folder=DRIVE_EXPORT_FOLDER,
                                    selectors=PREDICTION_BASE_COLUMNS + list(props),
                                    dry_run=DRY_RUN,
                                )
                            )

    if manifest_rows:
        manifest_path = PROJECT_DIR / 'outputs' / 'gee_prediction_grid_covariate_export_manifest.csv'
        covariates.write_manifest(manifest_rows, manifest_path)
        print(f'Wrote prediction export manifest: {manifest_path}')
        print('Pause now: wait for all Earth Engine tasks to finish, then let Google Drive for Desktop sync the CSVs into:')
        print(DOWNLOADED_EXPORT_DIR)
    else:
        print('No new prediction exports submitted; all requested CSVs already exist in the synced Drive folder.')
else:
    print('RUN_DRIVE_EXPORTS is False; skipping prediction covariate exports.')


RUN_DRIVE_EXPORTS is False; skipping prediction covariate exports.


## Step 7: Merge Synced Prediction CSVs

Run this after all Earth Engine tasks finish and Google Drive for Desktop has synced the CSVs to `DOWNLOADED_EXPORT_DIR`. The merged output is written to `data/prediction_grid_covariates_2020_2025.csv`.


In [12]:
def read_export_tables(download_dir):
    csv_paths = sorted(Path(download_dir).glob('*.csv'))
    grouped = defaultdict(list)
    for path in csv_paths:
        prefix = normalized_export_prefix(path)
        if prefix.startswith(f'{PREDGRID_EXPORT_PREFIX}_'):
            grouped[prefix].append(path)
    tables = {}
    for prefix, paths in grouped.items():
        frames = [pd.read_csv(path) for path in paths]
        tables[prefix] = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
    return tables


def clean_prediction_table(table, value_columns):
    table = table.copy()
    for column in value_columns:
        if column in table.columns:
            table[column] = table[column].replace(covariates.NO_DATA_VALUE, pd.NA)
    return table


def expected_drive_units(groups, rings, years):
    """Group expected Drive exports by covariate table before merging.

    Per-year exports for the same covariate have the same value column names, so
    they must be row-bound across years before joining to the full grid-year base.
    """
    units = []
    year_values = [None] if EXPORT_ALL_YEARS_TOGETHER else years
    for group in groups:
        if is_nonscaled_export_group(group):
            names = [
                drive_description(group, year=year, batch_id=batch_id)
                for batch_id in grid_batch_ids()
                for year in year_values
            ]
            units.append((group, names))
        else:
            for ring in rings:
                names = [
                    drive_description(group, ring, year=year, batch_id=batch_id)
                    for batch_id in grid_batch_ids()
                    for year in year_values
                ]
                units.append((f'{group}_{ring}', names))
    return units


if MERGE_DOWNLOADED_FILES:
    require_google_drive_folder(DOWNLOADED_EXPORT_DIR, DRIVE_EXPORT_FOLDER)
    expected_units = expected_drive_units(GROUPS_TO_EXPORT, selected_rings, PREDICTION_YEARS)
    expected = [name for _, names in expected_units for name in names]
    tables = read_export_tables(DOWNLOADED_EXPORT_DIR)
    missing = sorted(set(expected).difference(tables))
    extra = sorted(set(tables).difference(expected))
    if missing:
        print('Missing expected downloaded CSV prefixes:')
        for name in missing:
            print('  ', name)
        raise FileNotFoundError('Downloaded prediction export folder is missing one or more expected CSVs.')
    if extra:
        print('Ignoring unexpected prediction CSV prefixes:')
        for name in extra:
            print('  ', name)

    base_frames = []
    for name in expected:
        table = tables[name]
        present_base = [column for column in PREDICTION_BASE_COLUMNS if column in table.columns]
        base_frames.append(table[present_base])
    final = (
        pd.concat(base_frames, ignore_index=True)
        .drop_duplicates(subset=['grid_id', 'year'])
        [PREDICTION_BASE_COLUMNS]
        .copy()
    )

    for unit_label, names in expected_units:
        unit_frames = []
        unit_value_columns = []
        for name in names:
            table = tables[name]
            drop_cols = {'.geo', 'system:index'}
            value_columns = [
                column for column in table.columns
                if column not in PREDICTION_BASE_COLUMNS and column not in drop_cols
            ]
            if not value_columns:
                print(f'No covariate columns found in {name}; skipping')
                continue
            table = clean_prediction_table(table, value_columns)
            for column in value_columns:
                if column not in unit_value_columns:
                    unit_value_columns.append(column)
            unit_frames.append(table[['grid_id', 'year'] + value_columns])

        if not unit_frames:
            print(f'No covariate columns found for {unit_label}; skipping')
            continue

        unit_table = pd.concat(unit_frames, ignore_index=True)
        if VALIDATE_DOWNLOADED_EXPORTS:
            blank_columns = [column for column in unit_value_columns if unit_table[column].notna().sum() == 0]
            if blank_columns:
                raise ValueError(
                    f'{unit_label} has all-blank exported covariate columns: {blank_columns}. '
                    'Delete those CSVs from the synced Drive folder and rerun the export after fixing the source issue.'
                )
        duplicate_columns = sorted(set(final.columns).intersection(unit_value_columns))
        if duplicate_columns:
            raise ValueError(f'Duplicate covariate columns while merging {unit_label}: {duplicate_columns}')
        final = final.merge(
            unit_table[['grid_id', 'year'] + unit_value_columns].drop_duplicates(subset=['grid_id', 'year']),
            on=['grid_id', 'year'],
            how='left',
        )

    final = final.sort_values(['year', 'y', 'x']).reset_index(drop=True)
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    final.to_csv(OUTPUT_CSV, index=False)
    final[PREDICTION_GRID_COLUMNS].drop_duplicates(subset=['grid_id']).sort_values(['y', 'x']).to_csv(OUTPUT_GRID_CSV, index=False)
    print(f'Wrote merged prediction covariates: {OUTPUT_CSV}')
    print(f'Wrote unique prediction grid: {OUTPUT_GRID_CSV}')
    print(final.shape)
else:
    print('MERGE_DOWNLOADED_FILES is False; skipping prediction CSV merge.')


FileNotFoundError: Google Drive sync folder is missing: /Users/boazbaliejukia/Library/CloudStorage/GoogleDrive-baliejukiabaliesima@gmail.com/My Drive/Event_ENM_Prediction_Exports
Install Google Drive for Desktop and sync My Drive/Event_ENM_Prediction_Exports before merge/export.

## Step 8: Training Dataset Setup

This section uses the same active covariate definitions as the prediction grid, but reads the observed/pseudo-absence points from `dataset1.csv`. The training export prefix is intentionally separate from the standalone `02` notebook prefix to avoid mixing old four-ring or meter-buffer outputs with this three-ring combined workflow.


In [13]:
if not TRAINING_DATASET1_CSV.exists():
    raise FileNotFoundError(f'Could not find training dataset1: {TRAINING_DATASET1_CSV}')

training_dataset = pd.read_csv(TRAINING_DATASET1_CSV)
training_dataset['year'] = training_dataset['year'].astype(int)

missing_base_columns = [column for column in TRAINING_BASE_COLUMNS if column not in training_dataset.columns]
if missing_base_columns:
    raise ValueError(f'dataset1 is missing required training columns: {missing_base_columns}')


def resolve_training_years(dataset, years):
    present = sorted(dataset['year'].dropna().astype(int).unique().tolist())
    if years is None:
        return present
    selected = sorted({int(year) for year in years})
    missing = sorted(set(selected).difference(present))
    if missing:
        raise ValueError(f'Requested training years are not present in dataset1: {missing}')
    return selected


training_selected_years = resolve_training_years(training_dataset, TRAINING_YEARS_TO_EXPORT)
training_selected_rings = [suffix for _, _, suffix in covariates.selected_scaled_rings(TRAINING_BUFFER_RINGS)]

if 'land_mask_region' not in globals():
    raw_land_mask_region = covariates.load_study_region(
        path=STUDY_AREA_FILE,
        bbox=STUDY_AREA_BBOX,
        lat_buffer_degrees=STUDY_AREA_CONTEXT_BUFFER_DEGREES,
        lon_buffer_degrees=STUDY_AREA_CONTEXT_BUFFER_DEGREES,
    )
    land_mask_region = covariates.simplify_study_region(raw_land_mask_region, STUDY_REGION_SIMPLIFY_TOLERANCE_M)
if 'land_mask' not in globals():
    land_mask = covariates.land_mask_image(land_mask_region)

print(f'training dataset1 rows: {len(training_dataset):,}')
print(f'training years: {training_selected_years[0]}-{training_selected_years[-1]} ({len(training_selected_years)} years)')
print(f'training rings: {training_selected_rings}')
for inner_deg, outer_deg, suffix in covariates.selected_scaled_rings(TRAINING_BUFFER_RINGS):
    tolerance_deg = covariates.buffer_geometry_error_degrees(outer_deg)
    scale_deg = covariates.ring_scale_degrees(suffix)
    print(
        f'  {suffix}: {inner_deg:.3f}-{outer_deg:.3f} degrees '
        f'(~{covariates.approx_degrees_to_km(inner_deg):.1f}-{covariates.approx_degrees_to_km(outer_deg):.1f} km) | '
        f'simplify tolerance {tolerance_deg:.4f} degrees (~{covariates.approx_degrees_to_m(tolerance_deg):,} m) | '
        f'Hansen degree scale {scale_deg:.4f} (~{covariates.approx_degrees_to_m(scale_deg):,} m)'
    )
print(f'training groups: {GROUPS_TO_EXPORT}')
print(f'training buffer asset root: {TRAINING_BUFFER_ASSET_ROOT}')
print(f'training Drive export folder: {TRAINING_DRIVE_EXPORT_FOLDER}')
print(f'training Drive export prefix: {TRAINING_EXPORT_PREFIX}')
print(f'training download/merge folder: {TRAINING_DOWNLOADED_EXPORT_DIR}')
print(f'training export all years together: {TRAINING_EXPORT_ALL_YEARS_TOGETHER}')
print(f'training skip existing synced Drive CSVs: {TRAINING_SKIP_EXISTING_DRIVE_EXPORTS}')


training dataset1 rows: 5,026
training years: 2020-2020 (1 years)
training rings: ['0_10km', '10_25km', '25_50km']
  0_10km: 0.000-0.090 degrees (~0.0-10.0 km) | simplify tolerance 0.0018 degrees (~200 m) | Hansen degree scale 0.0018 (~200 m)
  10_25km: 0.090-0.225 degrees (~10.0-25.0 km) | simplify tolerance 0.0045 degrees (~501 m) | Hansen degree scale 0.0045 (~501 m)
  25_50km: 0.225-0.449 degrees (~25.0-50.0 km) | simplify tolerance 0.0270 degrees (~3,006 m) | Hansen degree scale 0.0180 (~2,004 m)
training groups: ['forest_cover', 'forest_loss_same_year', 'forest_loss_1yr_prior', 'forest_loss_2yr_prior', 'fragmentation', 'population', 'nonscaled_precip', 'nonscaled_temp', 'nonscaled_pet', 'nonscaled_ndvi', 'nonscaled_elevation']
training buffer asset root: projects/flu-landscape/assets/FiloPrediction/Event_ENM_training_degree_donut_buffers_simplified
training Drive export folder: Event_ENM_Exports
training Drive export prefix: dataset2_0203_deg_drive
training download/merge folder:

## Step 9: Training Helper Functions

These wrappers use the same source-image and reduction functions used above for prediction-grid extraction. Only the table key, base columns, asset IDs, and export descriptions differ.


In [14]:
import time

def training_base_columns(dataset):
    return [column for column in TRAINING_BASE_COLUMNS if column in dataset.columns]


def training_base_table(dataset):
    return dataset[training_base_columns(dataset)].drop_duplicates(subset=[TRAINING_MERGE_KEY]).copy()


def training_buffer_asset_id(ring_suffix):
    return covariates.training_buffer_asset_id(TRAINING_BUFFER_ASSET_ROOT, ring_suffix)


def training_buffer_asset_ids(rings):
    return {suffix: training_buffer_asset_id(suffix) for suffix in rings}


def training_buffer_features_for_year(ring_suffix, year, limit=None):
    features = ee.FeatureCollection(training_buffer_asset_id(ring_suffix)).filter(ee.Filter.eq('year', int(year)))
    if limit is not None:
        features = features.limit(int(limit))
    return features


def reduce_image_over_training_buffers(image, features, band_names, ring_suffix, scale_m):
    return reduce_image_over_prediction_buffers(image, features, band_names, ring_suffix, scale_m)


def extract_scaled_training_year(year, group_name, ring_suffix, limit=None):
    if group_name not in SCALED_GROUP_BANDS:
        raise ValueError(f'Unknown scaled group: {group_name}')
    image, band_names, scale_m = scaled_source_image(year, group_name, ring_suffix)
    features = training_buffer_features_for_year(ring_suffix, year, limit=limit)
    return reduce_image_over_training_buffers(image, features, band_names, ring_suffix, scale_m)


def extract_nonscaled_training_year(year, group_name, limit=None):
    groups = nonscaled_image_groups_for_export(ee.Number(int(year)), group_name)
    if not groups:
        raise ValueError(f'No nonscaled image groups matched {group_name}.')
    features = training_buffer_features_for_year('0_10km', year, limit=limit)
    collections = []
    output_names = []
    for group in groups:
        collection, props = reduce_image_over_training_buffers(
            group.image,
            features,
            group.band_names,
            '0_10km',
            group.scale_m,
        )
        collections.append((collection, props))
        output_names.extend(props)
    if len(collections) == 1:
        return collections[0][0], output_names
    return covariates.merge_ring_feature_collections(collections, join_key=TRAINING_MERGE_KEY), output_names


def training_normalized_export_prefix(path):
    return normalized_export_prefix(path)


def existing_training_drive_export_prefixes(download_dir):
    download_path = Path(download_dir)
    if not download_path.exists():
        print(f'Training Drive sync folder not found yet: {download_path}')
        return set()
    prefixes = set()
    for path in download_path.glob('*.csv'):
        prefix = training_normalized_export_prefix(path)
        if prefix.startswith(f'{TRAINING_EXPORT_PREFIX}_'):
            prefixes.add(prefix)
    return prefixes


def training_drive_description(group_name, ring_suffix=None, year=None):
    year_part = 'all_years' if year is None else str(int(year))
    if ring_suffix is None:
        return f'{TRAINING_EXPORT_PREFIX}_{year_part}_{group_name}'
    return f'{TRAINING_EXPORT_PREFIX}_{year_part}_{group_name}_{ring_suffix}'


def expected_training_drive_units(groups, rings, years):
    units = []
    year_values = [None] if TRAINING_EXPORT_ALL_YEARS_TOGETHER else years
    for group in groups:
        if is_nonscaled_export_group(group):
            names = [training_drive_description(group, year=year) for year in year_values]
            units.append((group, names))
        else:
            for ring in rings:
                names = [training_drive_description(group, ring, year=year) for year in year_values]
                units.append((f'{group}_{ring}', names))
    return units


def training_export_path_groups(download_dir):
    download_path = Path(download_dir)
    if not download_path.exists():
        raise FileNotFoundError(f'Training Drive sync folder does not exist: {download_path}')
    csv_paths = sorted(download_path.glob('*.csv'))
    grouped = defaultdict(list)
    for path in csv_paths:
        prefix = training_normalized_export_prefix(path)
        if prefix.startswith(f'{TRAINING_EXPORT_PREFIX}_'):
            grouped[prefix].append(path)
    return dict(grouped)


def read_csv_with_drive_retry(path, attempts=None, wait_seconds=None):
    attempts = int(attempts or TRAINING_CSV_READ_ATTEMPTS)
    wait_seconds = int(wait_seconds or TRAINING_CSV_READ_WAIT_SECONDS)
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return pd.read_csv(path)
        except OSError as exc:
            last_error = exc
            if attempt < attempts:
                print(f'Could not read {path}; retrying in {wait_seconds} seconds ({attempt}/{attempts}). Error: {exc}')
                time.sleep(wait_seconds)
    raise OSError(
        f'Could not read synced Drive CSV after {attempts} attempts: {path}. '
        'This usually means Google Drive for Desktop has not finished syncing the file, '
        'the file is cloud-only/unavailable offline, or the file is corrupted/truncated. '
        'Wait for sync to finish or mark the export folder available offline, then rerun the merge cell.'
    ) from last_error


def read_training_export_tables(path_groups, expected_prefixes):
    expected_prefixes = list(expected_prefixes)
    tables = {}
    for index, prefix in enumerate(expected_prefixes, start=1):
        paths = path_groups[prefix]
        frames = [read_csv_with_drive_retry(path) for path in paths]
        tables[prefix] = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
        if index % 5 == 0 or index == len(expected_prefixes):
            print(f'Read {index}/{len(expected_prefixes)} expected training export table(s).')
    return tables


def clean_training_table(table, value_columns):
    table = table.copy()
    for column in value_columns:
        if column in table.columns:
            table[column] = table[column].replace(covariates.NO_DATA_VALUE, pd.NA)
    return table


## Step 10: Create Training Buffer Assets

Run this when the training buffer assets do not exist or when the shared buffer definitions change. After submitting these asset tasks, wait for Earth Engine to finish them before continuing to the smoke test or Drive exports.


In [15]:
if TRAINING_CREATE_BUFFER_ASSETS:
    training_points = covariates.dataframe_to_feature_collection(training_dataset)
    study_region_for_training_buffers = (
        land_mask_region
        if TRAINING_CLIP_BUFFERS_TO_STUDY_AREA
        else None
    )
    buffer_rows = []
    if TRAINING_CREATE_ASSET_FOLDER:
        buffer_rows.extend(covariates.ensure_ee_folder(TRAINING_BUFFER_ASSET_ROOT, dry_run=DRY_RUN))

    for _, outer_deg, ring_suffix in covariates.selected_scaled_rings(TRAINING_BUFFER_RINGS):
        tolerance_deg = covariates.buffer_geometry_error_degrees(outer_deg)
        print(
            f'Training buffer {ring_suffix}: outer radius {outer_deg:.3f} degrees '
            f'(~{covariates.approx_degrees_to_km(outer_deg):.1f} km); '
            f'simplify tolerance {tolerance_deg:.4f} degrees '
            f'(~{covariates.approx_degrees_to_m(tolerance_deg):,} m)'
        )
        buffers = covariates.make_training_buffer_collection(
            training_points,
            ring_suffix,
            study_region=study_region_for_training_buffers,
        )
        buffer_rows.append(
            covariates.export_table_to_asset(
                collection=buffers,
                description=f'training_buffers_{ring_suffix}',
                asset_id=training_buffer_asset_id(ring_suffix),
                dry_run=DRY_RUN,
                overwrite=TRAINING_OVERWRITE_BUFFER_ASSETS,
            )
        )

    manifest_path = PROJECT_DIR / 'outputs' / 'gee_0203_training_buffer_asset_manifest.csv'
    covariates.write_manifest(buffer_rows, manifest_path)
    print(f'Wrote training buffer manifest: {manifest_path}')
    print('Pause now: wait for all training buffer asset tasks to finish before continuing.')
else:
    print('TRAINING_CREATE_BUFFER_ASSETS is False; using existing training buffer assets.')


EEException: Permission 'earthengine.assets.create' denied on resource 'projects/flu-landscape' (or it may not exist).

## Step 11: Training Smoke Test Extraction

This runs a tiny extraction against the training buffer assets. It is optional but useful after changing sources, ring definitions, or asset roots.


In [ ]:
if TRAINING_RUN_GEE_SMOKE_TEST:
    if TRAINING_CREATE_BUFFER_ASSETS:
        print(
            'TRAINING_CREATE_BUFFER_ASSETS is True. Continuing because this workflow assumes you manually waited '
            'for the training buffer asset tasks to finish before running this cell.'
        )

    smoke_year = int(TRAINING_SMOKE_TEST_YEAR or (2020 if 2020 in training_selected_years else training_selected_years[-1]))
    if smoke_year not in training_selected_years:
        raise ValueError(f'TRAINING_SMOKE_TEST_YEAR {smoke_year} is not present in training years.')

    smoke_rows = []
    smoke_groups = ['forest_cover', 'nonscaled_precip']
    smoke_rings = ['0_10km']
    print(f'Training smoke test groups: {smoke_groups}')
    print(f'Training smoke test rings: {smoke_rings}')
    print(f'Training smoke test features per extract: {TRAINING_SMOKE_TEST_MAX_FEATURES}')

    for group_name in smoke_groups:
        if is_nonscaled_export_group(group_name):
            collection, props = extract_nonscaled_training_year(
                smoke_year,
                group_name,
                limit=TRAINING_SMOKE_TEST_MAX_FEATURES,
            )
            smoke_rows.append(
                summarize_nonnull_properties(
                    training_drive_description(group_name, year=smoke_year),
                    collection,
                    props,
                )
            )
        else:
            for ring_suffix in smoke_rings:
                collection, props = extract_scaled_training_year(
                    smoke_year,
                    group_name,
                    ring_suffix,
                    limit=TRAINING_SMOKE_TEST_MAX_FEATURES,
                )
                smoke_rows.append(
                    summarize_nonnull_properties(
                        training_drive_description(group_name, ring_suffix, smoke_year),
                        collection,
                        props,
                    )
                )

    training_smoke_summary = pd.DataFrame(smoke_rows)
    display(training_smoke_summary)
    failed = training_smoke_summary.loc[training_smoke_summary['blank_columns'].astype(str).ne('')]
    if TRAINING_FAIL_IF_SMOKE_TEST_BLANK and not failed.empty:
        raise ValueError('Training smoke test found all-blank covariate columns. Review the table above before exporting.')
else:
    print('TRAINING_RUN_GEE_SMOKE_TEST is False; skipping training smoke test.')


## Step 12: Submit Training Covariate Drive Exports

This submits the training covariate CSV exports to Google Drive. If `TRAINING_EXPORT_ALL_YEARS_TOGETHER` is true, the workflow creates one CSV per covariate group/ring across all training years, which is 23 expected files with the current three-ring setup.


In [ ]:
if TRAINING_RUN_DRIVE_EXPORTS:
    # Local sync folder must exist so completed Drive exports can be detected/merged.
    require_google_drive_folder(TRAINING_DOWNLOADED_EXPORT_DIR, TRAINING_DRIVE_EXPORT_FOLDER)
    if TRAINING_CREATE_BUFFER_ASSETS:
        print(
            'TRAINING_CREATE_BUFFER_ASSETS is True. Continuing because this workflow assumes you manually waited '
            'for the training buffer asset tasks to finish before submitting training Drive covariate exports.'
        )

    existing_prefixes = existing_training_drive_export_prefixes(TRAINING_DOWNLOADED_EXPORT_DIR) if TRAINING_SKIP_EXISTING_DRIVE_EXPORTS else set()
    if TRAINING_SKIP_EXISTING_DRIVE_EXPORTS:
        print(f'Found {len(existing_prefixes)} existing synced training export(s) to skip.')

    def should_submit_training_export(description):
        if description in existing_prefixes:
            print(f'{description} | synced CSV exists; skipping export')
            return False
        return True

    manifest_rows = []
    for group_name in GROUPS_TO_EXPORT:
        if is_nonscaled_export_group(group_name):
            if TRAINING_EXPORT_ALL_YEARS_TOGETHER:
                description = training_drive_description(group_name)
                if not should_submit_training_export(description):
                    continue
                year_collections = []
                props = None
                for year in training_selected_years:
                    collection, props = extract_nonscaled_training_year(year, group_name)
                    year_collections.append(collection)
                manifest_rows.append(
                    covariates.export_table_to_drive(
                        collection=merge_year_collections(year_collections),
                        description=description,
                        folder=TRAINING_DRIVE_EXPORT_FOLDER,
                        selectors=training_base_columns(training_dataset) + list(props or []),
                        dry_run=DRY_RUN,
                    )
                )
            else:
                for year in training_selected_years:
                    description = training_drive_description(group_name, year=year)
                    if not should_submit_training_export(description):
                        continue
                    collection, props = extract_nonscaled_training_year(year, group_name)
                    manifest_rows.append(
                        covariates.export_table_to_drive(
                            collection=collection,
                            description=description,
                            folder=TRAINING_DRIVE_EXPORT_FOLDER,
                            selectors=training_base_columns(training_dataset) + list(props),
                            dry_run=DRY_RUN,
                        )
                    )
        else:
            if group_name not in SCALED_GROUP_BANDS:
                raise ValueError(f'Unknown scaled group: {group_name}')
            for ring_suffix in training_selected_rings:
                if TRAINING_EXPORT_ALL_YEARS_TOGETHER:
                    description = training_drive_description(group_name, ring_suffix)
                    if not should_submit_training_export(description):
                        continue
                    year_collections = []
                    props = None
                    for year in training_selected_years:
                        collection, props = extract_scaled_training_year(year, group_name, ring_suffix)
                        year_collections.append(collection)
                    manifest_rows.append(
                        covariates.export_table_to_drive(
                            collection=merge_year_collections(year_collections),
                            description=description,
                            folder=TRAINING_DRIVE_EXPORT_FOLDER,
                            selectors=training_base_columns(training_dataset) + list(props or []),
                            dry_run=DRY_RUN,
                        )
                    )
                else:
                    for year in training_selected_years:
                        description = training_drive_description(group_name, ring_suffix, year=year)
                        if not should_submit_training_export(description):
                            continue
                        collection, props = extract_scaled_training_year(year, group_name, ring_suffix)
                        manifest_rows.append(
                            covariates.export_table_to_drive(
                                collection=collection,
                                description=description,
                                folder=TRAINING_DRIVE_EXPORT_FOLDER,
                                selectors=training_base_columns(training_dataset) + list(props),
                                dry_run=DRY_RUN,
                            )
                        )

    if manifest_rows:
        manifest_path = PROJECT_DIR / 'outputs' / 'gee_0203_training_covariate_export_manifest.csv'
        covariates.write_manifest(manifest_rows, manifest_path)
        print(f'Wrote training export manifest: {manifest_path}')
        print('Pause now: wait for all Earth Engine tasks to finish, then let Google Drive for Desktop sync the CSVs into:')
        print(TRAINING_DOWNLOADED_EXPORT_DIR)
    else:
        print('No new training exports submitted; all requested CSVs already exist in the synced Drive folder.')
else:
    print('TRAINING_RUN_DRIVE_EXPORTS is False; skipping training covariate exports.')


## Step 13: Merge Synced Training CSVs

Run this after the training Earth Engine tasks finish and Google Drive for Desktop has synced the CSVs into `TRAINING_DOWNLOADED_EXPORT_DIR`. The merged training output is written to `data/dataset2.csv`.


In [ ]:
if TRAINING_MERGE_DOWNLOADED_FILES:
    require_google_drive_folder(TRAINING_DOWNLOADED_EXPORT_DIR, TRAINING_DRIVE_EXPORT_FOLDER)
    expected_units = expected_training_drive_units(GROUPS_TO_EXPORT, training_selected_rings, training_selected_years)
    expected = [name for _, names in expected_units for name in names]
    path_groups = training_export_path_groups(TRAINING_DOWNLOADED_EXPORT_DIR)
    available = set(path_groups)
    missing = sorted(set(expected).difference(available))
    extra = sorted(available.difference(expected))
    if missing:
        print('Missing expected downloaded training CSV prefixes:')
        for name in missing:
            print('  ', name)
        raise FileNotFoundError('Downloaded training export folder is missing one or more expected Drive CSVs.')
    if extra:
        print('Ignoring unexpected training Drive CSV prefixes:')
        for name in extra:
            print('  ', name)

    tables = read_training_export_tables(path_groups, expected)

    final_training = training_base_table(training_dataset)
    if len(final_training) != len(training_dataset):
        raise ValueError(
            f'Training merge key {TRAINING_MERGE_KEY!r} is not unique: '
            f'{len(final_training):,} unique rows for {len(training_dataset):,} input rows.'
        )

    for unit_label, names in expected_units:
        unit_frames = []
        unit_value_columns = []
        for name in names:
            table = tables[name]
            drop_cols = {'.geo', 'system:index'}
            value_columns = [
                column for column in table.columns
                if column not in TRAINING_BASE_COLUMNS and column not in drop_cols and column != TRAINING_MERGE_KEY
            ]
            if not value_columns:
                print(f'No covariate columns found in {name}; skipping')
                continue
            table = clean_training_table(table, value_columns)
            for column in value_columns:
                if column not in unit_value_columns:
                    unit_value_columns.append(column)
            unit_frames.append(table[[TRAINING_MERGE_KEY] + value_columns])

        if not unit_frames:
            print(f'No covariate columns found for {unit_label}; skipping')
            continue

        unit_table = pd.concat(unit_frames, ignore_index=True)
        if TRAINING_VALIDATE_DOWNLOADED_EXPORTS:
            blank_columns = [column for column in unit_value_columns if unit_table[column].notna().sum() == 0]
            if blank_columns:
                raise ValueError(
                    f'{unit_label} has all-blank exported covariate columns: {blank_columns}. '
                    'Delete those CSVs from the synced Drive folder and rerun the export after fixing the source issue.'
                )
        duplicate_columns = sorted(set(final_training.columns).intersection(unit_value_columns))
        if duplicate_columns:
            raise ValueError(f'Duplicate covariate columns while merging {unit_label}: {duplicate_columns}')
        final_training = final_training.merge(
            unit_table[[TRAINING_MERGE_KEY] + unit_value_columns].drop_duplicates(subset=[TRAINING_MERGE_KEY]),
            on=TRAINING_MERGE_KEY,
            how='left',
        )

    final_training = final_training.sort_values(['year', 'id']).reset_index(drop=True)
    if len(final_training) != len(training_dataset):
        raise ValueError(
            f'Merged training row count changed: {len(final_training):,} rows after merge, '
            f'but dataset1 has {len(training_dataset):,} rows.'
        )

    training_covariate_columns = [column for column in final_training.columns if column not in TRAINING_BASE_COLUMNS]
    if OUTPUT_CSV.exists():
        prediction_columns = list(pd.read_csv(OUTPUT_CSV, nrows=0).columns)
        prediction_covariate_columns = [column for column in prediction_columns if column not in PREDICTION_BASE_COLUMNS]
        if training_covariate_columns != prediction_covariate_columns:
            missing_from_training = [column for column in prediction_covariate_columns if column not in training_covariate_columns]
            extra_in_training = [column for column in training_covariate_columns if column not in prediction_covariate_columns]
            raise ValueError(
                'Training covariate columns do not match the prediction-grid covariate columns. '
                f'Missing from training: {missing_from_training}; extra in training: {extra_in_training}'
            )

    TRAINING_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    final_training.to_csv(TRAINING_OUTPUT_CSV, index=False)
    print(f'Wrote merged training dataset: {TRAINING_OUTPUT_CSV}')
    print(final_training.shape)
else:
    print('TRAINING_MERGE_DOWNLOADED_FILES is False; skipping training CSV merge.')
